# Mốc 8, bước 4: fine-tune model đọc chữ cho tiếng Việt (cần GPU)

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU** → Save.

Cách làm: lấy model **latin PP-OCRv5 mobile** đã train sẵn, **thay lớp cuối** (503 ký tự latin → 236 ký tự tiếng Việt), học tiếp trên 60.000 dòng tiếng Việt (dữ liệu của bước 3 trên Google Drive).
Các lớp có kích thước khớp được nạp từ trọng số gốc; lớp cuối (không khớp) được khởi tạo mới.

Kết quả (trọng số tốt nhất) lưu vào `MyDrive/doc-scanner-ocr/output/` nên Colab ngắt giữa chừng cũng không mất.
Thời gian dự kiến: khoảng 1 giờ (20 epoch).

In [ ]:
# 1. Kiểm tra GPU, kết nối Drive, giải nén dữ liệu
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/doc-scanner-ocr"
!cd /content && rm -rf data && unzip -q $OUT/data.zip && ls data | head && wc -l data/train.txt data/eval.txt data/vi_dict.txt

In [ ]:
# 2. Cài PaddlePaddle bản GPU + mã nguồn PaddleOCR (bản v3.7.0) (khoảng 3-5 phút)
!pip -q install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!git clone -q --depth 1 --branch v3.7.0 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
!pip -q install -r /content/PaddleOCR/requirements.txt
import paddle
paddle.utils.run_check()
# Nếu dòng cài paddlepaddle-gpu báo lỗi phiên bản: xem lệnh cài đúng cho CUDA của Colab tại https://www.paddlepaddle.org.cn/install/quick

In [ ]:
# 3. Tải trọng số gốc của model latin PP-OCRv5 mobile
!cd /content && wget -q -nc https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/latin_PP-OCRv5_mobile_rec_pretrained.pdparams && ls -la /content/*.pdparams

In [ ]:
# 4. Fine-tune. Các tham số quan trọng (ghi đè file cấu hình gốc):
#   character_dict_path: từ điển tiếng Việt 236 ký tự (+ khoảng trắng nhờ use_space_char)
#   pretrained_model:    trọng số latin v5 (lớp nào không khớp kích thước thì bỏ qua -> lớp cuối học mới)
#   epoch_num 20, learning_rate 0.0002 (nhỏ vì chỉ học tiếp), warmup 1 epoch
#   first_bs 64: cấu hình gốc 128 ảnh / lượt làm GPU T4 (15 GB) hết bộ nhớ
# Theo dõi chi tiết khi đang chạy: mở Terminal của Colab, gõ  tail -n 8 $OUT/output/train.log
%cd /content/PaddleOCR
!python tools/train.py -c configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml -o \
  Global.pretrained_model=/content/latin_PP-OCRv5_mobile_rec_pretrained \
  Global.character_dict_path=/content/data/vi_dict.txt \
  Global.save_model_dir=$OUT/output \
  Global.epoch_num=20 Global.save_epoch_step=5 Global.eval_batch_step=[0,1000] Global.distributed=false \
  Optimizer.lr.learning_rate=0.0002 Optimizer.lr.warmup_epoch=1 \
  Train.dataset.data_dir=/content/data/ Train.dataset.label_file_list=[/content/data/train.txt] \
  Train.sampler.first_bs=64 Train.loader.batch_size_per_card=64 Train.loader.num_workers=2 \
  Eval.dataset.data_dir=/content/data/ Eval.dataset.label_file_list=[/content/data/eval.txt] \
  Eval.loader.num_workers=2 2>&1 | grep --line-buffered -E "load pretrain|not matched|best metric|cur metric|Error|error|global_step: [0-9]*00,"

In [ ]:
# 5. Đánh giá trọng số tốt nhất trên tập eval (font NotoSerif chưa gặp khi train)
#    acc = tỉ lệ dòng đọc đúng hoàn toàn; norm_edit_dis = 1 - (tỉ lệ ký tự sai) -> CER = 1 - norm_edit_dis
!python tools/eval.py -c configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml -o \
  Global.pretrained_model=$OUT/output/best_accuracy Global.character_dict_path=/content/data/vi_dict.txt \
  Eval.dataset.data_dir=/content/data/ Eval.dataset.label_file_list=[/content/data/eval.txt] Eval.loader.num_workers=2 \
  2>&1 | grep -E "acc|norm_edit_dis|fps" | tail -5

In [ ]:
# 6. Đọc thử 15 ảnh eval đầu tiên: so chữ model đọc với đáp án
!rm -rf /tmp/sample && mkdir -p /tmp/sample && head -15 /content/data/eval.txt | cut -f1 | xargs -I{} cp /content/data/{} /tmp/sample/
!python tools/infer_rec.py -c configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml -o \
  Global.pretrained_model=$OUT/output/best_accuracy Global.character_dict_path=/content/data/vi_dict.txt \
  Global.infer_img=/tmp/sample 2>&1 | grep "result:"
print("--- dap an ---")
!head -15 /content/data/eval.txt